<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/06_LLM_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# 06.1 LOAD PROFILES
# ============================================================

from pathlib import Path
import json
import time
import re
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")

if not (
    DRIVE_MOUNT.exists()
    and (DRIVE_MOUNT / "MyDrive").exists()
):

    drive.mount(
        "/content/drive",
        force_remount=False
    )

# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"""
AIR-LLM project root not found:

{PROJECT_ROOT}

Ensure Google Drive is mounted and the
AIR_LLM_Research folder exists in MyDrive.
"""
    )

# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

# ------------------------------------------------------------
# PROFILE ARTIFACT PATHS
# ------------------------------------------------------------

PROFILE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "profiles"
)

DATASET_PROFILE_PATH = (
    PROFILE_ROOT /
    "dataset" /
    "dataset_profiles.csv"
)

FEATURE_PROFILE_PATH = (
    PROFILE_ROOT /
    "feature" /
    "feature_profiles.csv"
)

AIR_LLM_CONTEXT_PATH = (
    PROFILE_ROOT /
    "air_llm_context.json"
)

PROFILE_REGISTRY_PATH = (
    PROFILE_ROOT /
    "profile_registry.csv"
)

# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

RECOMMENDATION_ROOT = (
    PROJECT_ROOT /
    "results" /
    "recommendations"
)

RECOMMENDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# VERIFY REQUIRED FILES
# ------------------------------------------------------------

REQUIRED_PROFILE_FILES = {

    "dataset_profiles":
        DATASET_PROFILE_PATH,

    "feature_profiles":
        FEATURE_PROFILE_PATH,

    "air_llm_context":
        AIR_LLM_CONTEXT_PATH,

    "profile_registry":
        PROFILE_REGISTRY_PATH
}

missing_files = [
    str(path)
    for path in REQUIRED_PROFILE_FILES.values()
    if not path.exists()
]

if missing_files:

    raise FileNotFoundError(
        "Required Notebook 05 artifacts are missing:\n\n"
        + "\n".join(
            f"  - {path}"
            for path in missing_files
        )
    )

# ------------------------------------------------------------
# LOAD CSV ARTIFACTS
# ------------------------------------------------------------

DATASET_PROFILE_DF = pd.read_csv(
    DATASET_PROFILE_PATH
)

FEATURE_PROFILE_DF = pd.read_csv(
    FEATURE_PROFILE_PATH
)

PROFILE_REGISTRY_DF = pd.read_csv(
    PROFILE_REGISTRY_PATH
)

# ------------------------------------------------------------
# LOAD JSON CONTEXT
# ------------------------------------------------------------

with open(
    AIR_LLM_CONTEXT_PATH,
    "r",
    encoding="utf-8"
) as f:

    AIR_LLM_CONTEXT = json.load(f)

# ------------------------------------------------------------
# VALIDATE DATASET IDS
# ------------------------------------------------------------

assert set(DATASET_IDS).issubset(
    set(
        DATASET_PROFILE_DF[
            "dataset_id"
        ].astype(str)
    )
)

assert set(DATASET_IDS).issubset(
    set(
        FEATURE_PROFILE_DF[
            "dataset_id"
        ].astype(str)
    )
)

assert set(DATASET_IDS).issubset(
    set(AIR_LLM_CONTEXT.keys())
)

print("=" * 90)
print("NOTEBOOK 06 — PROFILES LOADED")
print("=" * 90)

print(
    f"Dataset profile rows : "
    f"{len(DATASET_PROFILE_DF)}"
)

print(
    f"Feature profile rows : "
    f"{len(FEATURE_PROFILE_DF)}"
)

print(
    f"Datasets loaded      : "
    f"{len(DATASET_IDS)}"
)

print(
    f"Recommendation root  : "
    f"{RECOMMENDATION_ROOT}"
)

NOTEBOOK 06 — PROFILES LOADED
Dataset profile rows : 3
Feature profile rows : 80
Datasets loaded      : 3
Recommendation root  : /content/drive/MyDrive/AIR_LLM_Research/results/recommendations


In [4]:
# ============================================================
# 06.2 LOAD CANDIDATE STRATEGY REGISTRY
# ============================================================

print("=" * 90)
print("NOTEBOOK 06 — CANDIDATE STRATEGY REGISTRY")
print("=" * 90)

CANDIDATE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "candidates"
)

CANDIDATE_REGISTRY_PATH = (
    CANDIDATE_ROOT /
    "candidate_strategy_registry.csv"
)

# ------------------------------------------------------------
# FALLBACK SEARCH
# ------------------------------------------------------------

if not CANDIDATE_REGISTRY_PATH.exists():

    possible_paths = [

        PROJECT_ROOT /
        "results" /
        "candidate_strategy_registry.csv",

        PROJECT_ROOT /
        "results" /
        "registry" /
        "candidate_strategy_registry.csv"
    ]

    discovered = [
        path
        for path in possible_paths
        if path.exists()
    ]

    if discovered:

        CANDIDATE_REGISTRY_PATH = (
            discovered[0]
        )

# ------------------------------------------------------------
# HARD VALIDATION
# ------------------------------------------------------------

if not CANDIDATE_REGISTRY_PATH.exists():

    raise FileNotFoundError(
        """
Notebook 04 candidate strategy registry
could not be found.

Expected canonical location:

results/candidates/
candidate_strategy_registry.csv

Run the final candidate-registry save
cell in Notebook 04 first.
"""
    )

# ------------------------------------------------------------
# LOAD REGISTRY
# ------------------------------------------------------------

CANDIDATE_STRATEGY_REGISTRY_DF = pd.read_csv(
    CANDIDATE_REGISTRY_PATH
)

required_registry_columns = {
    "method"
}

missing_registry_columns = (
    required_registry_columns
    -
    set(
        CANDIDATE_STRATEGY_REGISTRY_DF.columns
    )
)

assert not missing_registry_columns, (
    "Candidate registry is missing columns: "
    f"{missing_registry_columns}"
)

# ------------------------------------------------------------
# NORMALIZE METHOD NAMES
# ------------------------------------------------------------

CANDIDATE_STRATEGY_REGISTRY_DF[
    "method"
] = (
    CANDIDATE_STRATEGY_REGISTRY_DF[
        "method"
    ]
    .astype(str)
    .str.strip()
)

METHOD_NAMES = (
    CANDIDATE_STRATEGY_REGISTRY_DF[
        "method"
    ]
    .tolist()
)

assert len(METHOD_NAMES) > 0
assert len(METHOD_NAMES) == len(
    set(METHOD_NAMES)
)

print(
    f"Registry path : "
    f"{CANDIDATE_REGISTRY_PATH}"
)

print(
    f"Candidate methods : "
    f"{len(METHOD_NAMES)}"
)

for method in METHOD_NAMES:

    print(
        f"  - {method}"
    )

NOTEBOOK 06 — CANDIDATE STRATEGY REGISTRY
Registry path : /content/drive/MyDrive/AIR_LLM_Research/results/candidates/candidate_strategy_registry.csv
Candidate methods : 9
  - Mean
  - Median
  - Mode
  - KNN
  - MICE
  - MissForest
  - GAIN
  - LLM
  - AutomatedSelection


In [6]:
# ============================================================
# 06.3 CONSTRUCT STRUCTURED LLM INPUT
# ============================================================

TOP_K = 5

# ------------------------------------------------------------
# HELPER
# ------------------------------------------------------------

def json_safe(value):

    if isinstance(value, dict):

        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, list):

        return [
            json_safe(v)
            for v in value
        ]

    if isinstance(value, tuple):

        return [
            json_safe(v)
            for v in value
        ]

    if isinstance(
        value,
        (np.integer,)
    ):

        return int(value)

    if isinstance(
        value,
        (np.floating,)
    ):

        value = float(value)

        return (
            None
            if not np.isfinite(value)
            else value
        )

    if isinstance(
        value,
        (np.bool_,)
    ):

        return bool(value)

    if isinstance(
        value,
        float
    ):

        return (
            None
            if not np.isfinite(value)
            else value
        )

    return value


# ------------------------------------------------------------
# BUILD FEATURE-LEVEL INPUTS
# ------------------------------------------------------------

LLM_INPUTS = []

for dataset_id in DATASET_IDS:

    dataset_context = (
        AIR_LLM_CONTEXT[
            dataset_id
        ]
    )

    dataset_profile = (
        dataset_context[
            "dataset"
        ]
    )

    feature_profiles = (
        dataset_context[
            "incomplete_features"
        ]
    )

    for feature, profile in feature_profiles.items():

        missing_rate = (
            profile
            .get("M", {})
            .get("missing_rate", 0.0)
        )

        if missing_rate is None:
            continue

        try:
            missing_rate = float(
                missing_rate
            )
        except Exception:
            continue

        # ----------------------------------------------------
        # ONLY INCOMPLETE FEATURES
        # ----------------------------------------------------

        if missing_rate <= 0:
            continue

        structured_input = {

            "dataset_id":
                dataset_id,

            "dataset_profile":
                dataset_profile,

            "feature":
                feature,

            "feature_profile":
                profile,

            "candidate_methods":
                METHOD_NAMES,

            "top_k":
                TOP_K
        }

        LLM_INPUTS.append(
            json_safe(
                structured_input
            )
        )

LLM_INPUT_DF = pd.DataFrame({

    "dataset_id": [
        x["dataset_id"]
        for x in LLM_INPUTS
    ],

    "feature": [
        x["feature"]
        for x in LLM_INPUTS
    ],

    "missing_rate": [

        x[
            "feature_profile"
        ]
        .get("M", {})
        .get(
            "missing_rate",
            np.nan
        )

        for x in LLM_INPUTS
    ]
})

print("=" * 90)
print("STRUCTURED LLM INPUT CONSTRUCTED")
print("=" * 90)

print(
    f"Incomplete features : "
    f"{len(LLM_INPUTS)}"
)

display(
    LLM_INPUT_DF
)


STRUCTURED LLM INPUT CONSTRUCTED
Incomplete features : 9


,dataset_id,feature,missing_rate
0,diabetes_130us,race,0.022601
1,diabetes_130us,weight,0.968801
2,diabetes_130us,payer_code,0.396633
3,diabetes_130us,medical_specialty,0.490395
4,diabetes_130us,diag_1,0.000213
5,diabetes_130us,diag_2,0.003488
6,diabetes_130us,diag_3,0.014036
7,diabetes_130us,max_glu_serum,0.947657
8,diabetes_130us,A1Cresult,0.832998


In [7]:
# ============================================================
# 06.4 LLM SYSTEM PROMPT
# ============================================================

LLM_SYSTEM_PROMPT = """
You are AIR-LLM, an intelligent strategy recommendation
engine for missing-value imputation in heterogeneous
tabular datasets.

Your task is ONLY to recommend and rank candidate
imputation strategies.

You MUST NOT generate:
- imputed values
- synthetic rows
- replacement values
- completed datasets
- cell-level predictions

The actual imputation will be performed later by the
experimental pipeline.

Use the supplied dataset profile and feature profile as
the evidence base.

Candidate strategies are restricted to the supplied
candidate strategy registry.

Reason about:
1. feature type,
2. missingness rate,
3. distribution characteristics,
4. cardinality,
5. feature dependencies,
6. predictive relevance,
7. dataset scale,
8. computational suitability.

You must rank only strategies present in the candidate
registry.

The recommendation should be evidence-driven rather
than based on generic preferences.

Return ONLY valid JSON matching the requested schema.

The response must contain:
- feature
- candidate_1
- candidate_2
- candidate_3
- candidate_4
- candidate_5
- reason
- confidence

Candidates must be ordered from best to worst.

Confidence must be a number between 0 and 1.

Do not include strategies outside the candidate registry.
Do not return imputed values.
"""

print(
    "LLM system prompt constructed."
)

LLM system prompt constructed.


In [8]:
# ============================================================
# 06.5 FEATURE-LEVEL PROMPT
# ============================================================

def build_feature_prompt(
    structured_input
):

    return f"""
Recommend the best missing-value imputation strategies
for the following incomplete feature.

DATASET AND FEATURE EVIDENCE
----------------------------

{json.dumps(
    structured_input,
    indent=2,
    ensure_ascii=False
)}

CANDIDATE STRATEGY REGISTRY
---------------------------

{json.dumps(
    METHOD_NAMES,
    indent=2
)}

SELECTION REQUIREMENTS
----------------------

1. Select only strategies from the registry.
2. Rank the top {TOP_K} strategies.
3. Rank candidates from strongest to weakest.
4. Explain the recommendation using the supplied
   statistical and dependency evidence.
5. Consider computational scale.
6. Do not invent additional methods.
7. Do not generate any imputed values.
8. Confidence must be between 0 and 1.

Return JSON with exactly these fields:

{{
  "feature": "<feature name>",
  "candidate_1": "<method>",
  "candidate_2": "<method>",
  "candidate_3": "<method>",
  "candidate_4": "<method>",
  "candidate_5": "<method>",
  "reason": "<concise evidence-based explanation>",
  "confidence": 0.0
}}
"""

FEATURE_PROMPTS = [

    build_feature_prompt(
        item
    )

    for item in LLM_INPUTS
]

print(
    f"Feature-level prompts constructed: "
    f"{len(FEATURE_PROMPTS)}"
)

Feature-level prompts constructed: 9


In [10]:
# ============================================================
# 06.0 LLM API CONFIGURATION
# ============================================================

import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

OPENAI_API_KEY = None

# ------------------------------------------------------------
# 1. LOAD FROM GOOGLE COLAB SECRETS
# ------------------------------------------------------------

if userdata is not None:
    try:
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        OPENAI_API_KEY = None

# ------------------------------------------------------------
# 2. FALLBACK TO ENVIRONMENT VARIABLE
# ------------------------------------------------------------

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.getenv(
        "OPENAI_API_KEY"
    )

# ------------------------------------------------------------
# 3. VALIDATE
# ------------------------------------------------------------

if not OPENAI_API_KEY:
    raise RuntimeError(
        """
OPENAI_API_KEY is not available.

For Google Colab:
1. Open the left sidebar.
2. Select Secrets (key icon).
3. Add a secret named:

    OPENAI_API_KEY

4. Store your API key as the secret value.
5. Enable notebook access to the secret.
6. Re-run this cell.

Do NOT hard-code the API key inside the notebook.
"""
    )

# ------------------------------------------------------------
# 4. CONFIGURATION
# ------------------------------------------------------------

LLM_MODEL = "gpt-4o-mini"

LLM_TEMPERATURE = 0.0

LLM_MAX_OUTPUT_TOKENS = 1000

print("=" * 90)
print("NOTEBOOK 06 — LLM CONFIGURATION")
print("=" * 90)

print(
    "API key status : AVAILABLE"
)

print(
    f"LLM model      : {LLM_MODEL}"
)

print(
    f"Temperature    : {LLM_TEMPERATURE}"
)

print(
    f"Max output     : {LLM_MAX_OUTPUT_TOKENS}"
)

print("=" * 90)

NOTEBOOK 06 — LLM CONFIGURATION
API key status : AVAILABLE
LLM model      : gpt-4o-mini
Temperature    : 0.0
Max output     : 1000


In [18]:
# ============================================================
# 06.6 GENERATE CANDIDATE RECOMMENDATIONS — ROBUST VERSION
# ============================================================

# ------------------------------------------------------------
# INSTALL / IMPORT
# ------------------------------------------------------------

!pip -q install -U openai

import os
import time
import json
import random
import pandas as pd

from openai import OpenAI

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

LLM_MODEL = os.environ.get(
    "AIR_LLM_MODEL",
    "gpt-4o-mini"
)

LLM_TEMPERATURE = 0.0
LLM_MAX_OUTPUT_TOKENS = 1000

# Retry configuration
MAX_RETRIES = 5
INITIAL_RETRY_DELAY = 2.0
MAX_RETRY_DELAY = 30.0

# Delay between successful API calls
REQUEST_DELAY = 1.0

# ------------------------------------------------------------
# LOAD OPENAI API KEY
# ------------------------------------------------------------

OPENAI_API_KEY = None

try:

    from google.colab import userdata

    OPENAI_API_KEY = userdata.get(
        "OPENAI_API_KEY"
    )

except Exception:

    OPENAI_API_KEY = None

# ------------------------------------------------------------
# FALLBACK: ENVIRONMENT VARIABLE
# ------------------------------------------------------------

if not OPENAI_API_KEY:

    OPENAI_API_KEY = os.environ.get(
        "OPENAI_API_KEY"
    )

# ------------------------------------------------------------
# VALIDATE API KEY
# ------------------------------------------------------------

if not OPENAI_API_KEY:

    raise RuntimeError(
        """
OPENAI_API_KEY is not available.

Google Colab:
1. Open the Secrets panel.
2. Add a secret named OPENAI_API_KEY.
3. Enter your API key.
4. Enable notebook access.
5. Re-run this cell.

Do NOT hard-code the API key.
"""
    )

# ------------------------------------------------------------
# INITIALIZE CLIENT
# ------------------------------------------------------------

client = OpenAI(
    api_key=OPENAI_API_KEY
)

print("=" * 90)
print("NOTEBOOK 06 — LLM CONFIGURATION")
print("=" * 90)

print(f"API key status : AVAILABLE")
print(f"LLM model      : {LLM_MODEL}")
print(f"Temperature    : {LLM_TEMPERATURE}")
print(f"Max tokens     : {LLM_MAX_OUTPUT_TOKENS}")
print(f"Max retries    : {MAX_RETRIES}")
print(f"Request delay  : {REQUEST_DELAY}s")

# ------------------------------------------------------------
# VALIDATE REQUIRED OBJECTS
# ------------------------------------------------------------

required_objects = [
    "FEATURE_PROMPTS",
    "LLM_INPUTS",
    "LLM_SYSTEM_PROMPT"
]

for object_name in required_objects:

    if object_name not in globals():

        raise RuntimeError(
            f"""
{object_name} is not available.

Run the required previous Notebook 06 cell
before executing Notebook 06.6.
"""
        )

if len(FEATURE_PROMPTS) != len(LLM_INPUTS):

    raise RuntimeError(
        "FEATURE_PROMPTS and LLM_INPUTS have different lengths."
    )

# ------------------------------------------------------------
# STRUCTURED OUTPUT SCHEMA
# ------------------------------------------------------------

RECOMMENDATION_SCHEMA = {

    "type": "object",

    "properties": {

        "feature": {
            "type": "string"
        },

        "candidate_1": {
            "type": "string"
        },

        "candidate_2": {
            "type": "string"
        },

        "candidate_3": {
            "type": "string"
        },

        "candidate_4": {
            "type": "string"
        },

        "candidate_5": {
            "type": "string"
        },

        "reason": {
            "type": "string"
        },

        "confidence": {
            "type": "number"
        }
    },

    "required": [
        "feature",
        "candidate_1",
        "candidate_2",
        "candidate_3",
        "candidate_4",
        "candidate_5",
        "reason",
        "confidence"
    ],

    "additionalProperties": False
}

# ------------------------------------------------------------
# HELPER — CLASSIFY API ERROR
# ------------------------------------------------------------

def classify_api_error(exc):

    error_text = str(exc).lower()

    # Quota / billing
    if (
        "insufficient_quota" in error_text
        or "exceeded your current quota" in error_text
        or "billing" in error_text
        or "quota" in error_text
    ):

        return "quota_error"

    # Rate limiting
    if (
        "rate limit" in error_text
        or "rate_limit" in error_text
        or "too many requests" in error_text
        or "429" in error_text
    ):

        return "rate_limit"

    # Authentication
    if (
        "401" in error_text
        or "unauthorized" in error_text
        or "invalid api key" in error_text
    ):

        return "authentication_error"

    # Permission
    if (
        "403" in error_text
        or "permission" in error_text
        or "forbidden" in error_text
    ):

        return "permission_error"

    # Other
    return "other_error"


# ------------------------------------------------------------
# HELPER — GENERATE ONE RECOMMENDATION
# ------------------------------------------------------------

def generate_recommendation(
    prompt,
    system_prompt,
    model,
    max_retries=5
):

    last_error = None

    for attempt in range(
        max_retries
    ):

        try:

            response = client.responses.create(

                model=model,

                instructions=system_prompt,

                input=prompt,

                temperature=LLM_TEMPERATURE,

                max_output_tokens=LLM_MAX_OUTPUT_TOKENS,

                text={
                    "format": {
                        "type": "json_schema",
                        "name": "air_llm_recommendation",
                        "strict": True,
                        "schema": RECOMMENDATION_SCHEMA
                    }
                }
            )

            output_text = response.output_text

            if not output_text:

                raise RuntimeError(
                    "LLM returned an empty response."
                )

            return {
                "success": True,
                "output": output_text,
                "error": None,
                "error_type": None,
                "attempts": attempt + 1
            }

        except Exception as exc:

            last_error = exc

            error_type = classify_api_error(
                exc
            )

            # ------------------------------------------------
            # DO NOT RETRY QUOTA / AUTH / PERMISSION ERRORS
            # ------------------------------------------------

            if error_type in [
                "quota_error",
                "authentication_error",
                "permission_error"
            ]:

                return {
                    "success": False,
                    "output": None,
                    "error": str(exc),
                    "error_type": error_type,
                    "attempts": attempt + 1
                }

            # ------------------------------------------------
            # RETRY TEMPORARY RATE LIMITS
            # ------------------------------------------------

            if error_type == "rate_limit":

                delay = min(
                    INITIAL_RETRY_DELAY * (2 ** attempt),
                    MAX_RETRY_DELAY
                )

                # Add jitter
                delay += random.uniform(
                    0,
                    1
                )

                print(
                    f"      Rate limit detected. "
                    f"Retry {attempt + 1}/{max_retries} "
                    f"after {delay:.1f}s"
                )

                time.sleep(
                    delay
                )

                continue

            # ------------------------------------------------
            # RETRY OTHER TRANSIENT ERRORS
            # ------------------------------------------------

            if attempt < max_retries - 1:

                delay = min(
                    INITIAL_RETRY_DELAY * (2 ** attempt),
                    MAX_RETRY_DELAY
                )

                delay += random.uniform(
                    0,
                    1
                )

                print(
                    f"      Temporary error. "
                    f"Retry {attempt + 1}/{max_retries} "
                    f"after {delay:.1f}s"
                )

                time.sleep(
                    delay
                )

            else:

                break

    return {
        "success": False,
        "output": None,
        "error": str(last_error),
        "error_type": classify_api_error(
            last_error
        ),
        "attempts": max_retries
    }


# ------------------------------------------------------------
# GENERATE RECOMMENDATIONS
# ------------------------------------------------------------

RAW_LLM_RESPONSES = []

total_prompts = len(
    FEATURE_PROMPTS
)

print()
print("=" * 90)
print("GENERATING AIR-LLM CANDIDATE RECOMMENDATIONS")
print("=" * 90)

print(
    f"Total feature prompts: {total_prompts}"
)

for index, prompt in enumerate(
    FEATURE_PROMPTS
):

    structured_input = LLM_INPUTS[index]

    dataset_id = structured_input.get(
        "dataset_id",
        "unknown"
    )

    feature = structured_input.get(
        "feature",
        "unknown"
    )

    print()
    print(
        f"[{index + 1}/{total_prompts}] "
        f"{dataset_id} → {feature}"
    )

    start_time = time.perf_counter()

    result = generate_recommendation(
        prompt=prompt,
        system_prompt=LLM_SYSTEM_PROMPT,
        model=LLM_MODEL,
        max_retries=MAX_RETRIES
    )

    runtime = (
        time.perf_counter()
        - start_time
    )

    status = (
        "success"
        if result["success"]
        else "failed"
    )

    RAW_LLM_RESPONSES.append({

        "experiment_index":
            int(index),

        "dataset_id":
            dataset_id,

        "feature":
            feature,

        "raw_response":
            result["output"],

        "status":
            status,

        "error_type":
            result["error_type"],

        "error":
            result["error"],

        "attempts":
            int(result["attempts"]),

        "runtime_seconds":
            float(runtime)
    })

    if result["success"]:

        print(
            f"      Status: SUCCESS "
            f"({runtime:.2f}s)"
        )

    else:

        print(
            f"      Status: FAILED"
        )

        print(
            f"      Error type: "
            f"{result['error_type']}"
        )

        print(
            f"      Error: "
            f"{result['error']}"
        )

        # ----------------------------------------------------
        # STOP EARLY FOR ACCOUNT-LEVEL PROBLEMS
        # ----------------------------------------------------

        if result["error_type"] in [
            "quota_error",
            "authentication_error",
            "permission_error"
        ]:

            print()
            print(
                "Account/API configuration problem "
                "detected."
            )

            print(
                "Stopping further API requests "
                "to avoid unnecessary calls."
            )

            # Store remaining prompts as not attempted
            for remaining_index in range(
                index + 1,
                total_prompts
            ):

                remaining_input = LLM_INPUTS[
                    remaining_index
                ]

                RAW_LLM_RESPONSES.append({

                    "experiment_index":
                        int(remaining_index),

                    "dataset_id":
                        remaining_input.get(
                            "dataset_id",
                            "unknown"
                        ),

                    "feature":
                        remaining_input.get(
                            "feature",
                            "unknown"
                        ),

                    "raw_response":
                        None,

                    "status":
                        "not_attempted",

                    "error_type":
                        result["error_type"],

                    "error":
                        "Skipped because account/API "
                        "configuration problem was detected.",

                    "attempts":
                        0,

                    "runtime_seconds":
                        0.0
                })

            break

    # --------------------------------------------------------
    # DELAY BETWEEN REQUESTS
    # --------------------------------------------------------

    if result["success"]:

        time.sleep(
            REQUEST_DELAY
        )


# ------------------------------------------------------------
# CREATE RAW RESPONSE DATAFRAME
# ------------------------------------------------------------

LLM_RAW_DF = pd.DataFrame(
    RAW_LLM_RESPONSES
)

# ------------------------------------------------------------
# VALIDATE OUTPUT
# ------------------------------------------------------------

if not LLM_RAW_DF.empty:

    success_count = int(
        (
            LLM_RAW_DF["status"]
            == "success"
        ).sum()
    )

    failed_count = int(
        (
            LLM_RAW_DF["status"]
            == "failed"
        ).sum()
    )

    not_attempted_count = int(
        (
            LLM_RAW_DF["status"]
            == "not_attempted"
        ).sum()
    )

else:

    success_count = 0
    failed_count = 0
    not_attempted_count = 0

# ------------------------------------------------------------
# GENERATION SUMMARY
# ------------------------------------------------------------

print()
print("=" * 90)
print("LLM RECOMMENDATION GENERATION COMPLETE")
print("=" * 90)

print(
    f"Total requests       : {total_prompts}"
)

print(
    f"Successful            : {success_count}"
)

print(
    f"Failed                : {failed_count}"
)

print(
    f"Not attempted         : {not_attempted_count}"
)

# ------------------------------------------------------------
# ERROR SUMMARY
# ------------------------------------------------------------

if not LLM_RAW_DF.empty:

    print()
    print("ERROR SUMMARY")
    print("-" * 90)

    error_summary = (
        LLM_RAW_DF[
            LLM_RAW_DF["status"] != "success"
        ]
        .groupby(
            "error_type",
            dropna=False
        )
        .size()
        .reset_index(
            name="count"
        )
    )

    display(
        error_summary
    )

# ------------------------------------------------------------
# DISPLAY FAILED REQUESTS
# ------------------------------------------------------------

failed_rows = LLM_RAW_DF[
    LLM_RAW_DF["status"] == "failed"
]

if not failed_rows.empty:

    print()
    print("FAILED REQUESTS")
    print("-" * 90)

    display(
        failed_rows[
            [
                "dataset_id",
                "feature",
                "error_type",
                "attempts",
                "error"
            ]
        ]
    )

# ------------------------------------------------------------
# SAVE RAW RESULTS
# ------------------------------------------------------------

RAW_OUTPUT_PATH = (
    "air_llm_raw_recommendations.csv"
)

LLM_RAW_DF.to_csv(
    RAW_OUTPUT_PATH,
    index=False
)

print()
print(
    f"Raw recommendation results saved to:"
)

print(
    RAW_OUTPUT_PATH
)

# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

if success_count == total_prompts:

    print()
    print(
        "✓ ALL AIR-LLM RECOMMENDATION REQUESTS "
        "COMPLETED SUCCESSFULLY."
    )

elif failed_count > 0:

    print()
    print(
        "⚠ SOME AIR-LLM REQUESTS FAILED."
    )

    print(
        "Review error_type and error columns "
        "before continuing."
    )

else:

    print()
    print(
        "⚠ NO RECOMMENDATIONS WERE GENERATED."
    )

NOTEBOOK 06 — LLM CONFIGURATION
API key status : AVAILABLE
LLM model      : gpt-4o-mini
Temperature    : 0.0
Max tokens     : 1000
Max retries    : 5
Request delay  : 1.0s

GENERATING AIR-LLM CANDIDATE RECOMMENDATIONS
Total feature prompts: 9

[1/9] diabetes_130us → race
      Status: FAILED
      Error type: quota_error
      Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Account/API configuration problem detected.
Stopping further API requests to avoid unnecessary calls.

LLM RECOMMENDATION GENERATION COMPLETE
Total requests       : 9
Successful            : 0
Failed                : 1
Not attempted         : 8

ERROR SUMMARY
------------------------------------------------------------------------------------------


,error_type,count
0,quota_error,9



FAILED REQUESTS
------------------------------------------------------------------------------------------


,dataset_id,feature,error_type,attempts,error
0,diabetes_130us,race,quota_error,1,Error code: 429 - {'error': {'message': 'You e...



Raw recommendation results saved to:
air_llm_raw_recommendations.csv

⚠ SOME AIR-LLM REQUESTS FAILED.
Review error_type and error columns before continuing.


In [22]:
# ============================================================
# 06.7 — PARSE STRUCTURED LLM RESPONSES
# ============================================================

import json
import pandas as pd

print("=" * 90)
print("NOTEBOOK 06.7 — PARSE STRUCTURED LLM RESPONSES")
print("=" * 90)

# ------------------------------------------------------------
# VALIDATE INPUT
# ------------------------------------------------------------

if "LLM_RAW_DF" not in globals():

    raise RuntimeError(
        """
LLM_RAW_DF is not available.

Run Notebook 06.6 before executing Notebook 06.7.
"""
    )

# ------------------------------------------------------------
# REQUIRED OUTPUT FIELDS
# ------------------------------------------------------------

REQUIRED_FIELDS = [

    "feature",
    "candidate_1",
    "candidate_2",
    "candidate_3",
    "candidate_4",
    "candidate_5",
    "reason",
    "confidence"
]

# ------------------------------------------------------------
# PARSE FUNCTION
# ------------------------------------------------------------

def parse_llm_response(row):

    raw_response = row.get(
        "raw_response"
    )

    status = row.get(
        "status"
    )

    error_type = row.get(
        "error_type"
    )

    # --------------------------------------------------------
    # API DID NOT RETURN A RESPONSE
    # --------------------------------------------------------

    if status != "success":

        return {

            "parse_status":
                "not_parsed",

            "parse_error":
                row.get(
                    "error",
                    "API request failed."
                ),

            "response_object":
                None
        }

    # --------------------------------------------------------
    # EMPTY RESPONSE
    # --------------------------------------------------------

    if raw_response is None:

        return {

            "parse_status":
                "empty_response",

            "parse_error":
                "LLM returned no response.",

            "response_object":
                None
        }

    # --------------------------------------------------------
    # JSON PARSING
    # --------------------------------------------------------

    try:

        parsed = json.loads(
            raw_response
        )

    except Exception as exc:

        return {

            "parse_status":
                "invalid_json",

            "parse_error":
                str(exc),

            "response_object":
                None
        }

    # --------------------------------------------------------
    # OBJECT TYPE VALIDATION
    # --------------------------------------------------------

    if not isinstance(
        parsed,
        dict
    ):

        return {

            "parse_status":
                "invalid_object",

            "parse_error":
                "Response is not a JSON object.",

            "response_object":
                None
        }

    # --------------------------------------------------------
    # FIELD VALIDATION
    # --------------------------------------------------------

    missing_fields = [

        field
        for field in REQUIRED_FIELDS
        if field not in parsed
    ]

    if missing_fields:

        return {

            "parse_status":
                "missing_fields",

            "parse_error":
                "Missing fields: "
                + ", ".join(
                    missing_fields
                ),

            "response_object":
                parsed
        }

    # --------------------------------------------------------
    # SUCCESS
    # --------------------------------------------------------

    return {

        "parse_status":
            "success",

        "parse_error":
            None,

        "response_object":
            parsed
    }


# ------------------------------------------------------------
# PARSE ALL RESPONSES
# ------------------------------------------------------------

PARSED_RESULTS = []

for _, row in LLM_RAW_DF.iterrows():

    parsed_result = parse_llm_response(
        row
    )

    parsed_object = (
        parsed_result[
            "response_object"
        ]
    )

    record = {

        "experiment_index":
            row.get(
                "experiment_index"
            ),

        "dataset_id":
            row.get(
                "dataset_id"
            ),

        "feature":
            row.get(
                "feature"
            ),

        "api_status":
            row.get(
                "status"
            ),

        "api_error_type":
            row.get(
                "error_type"
            ),

        "parse_status":
            parsed_result[
                "parse_status"
            ],

        "parse_error":
            parsed_result[
                "parse_error"
            ]
    }

    # --------------------------------------------------------
    # ADD STRUCTURED FIELDS
    # --------------------------------------------------------

    if isinstance(
        parsed_object,
        dict
    ):

        for field in REQUIRED_FIELDS:

            record[field] = (
                parsed_object.get(
                    field
                )
            )

    else:

        for field in REQUIRED_FIELDS:

            record[field] = None

    PARSED_RESULTS.append(
        record
    )


# ------------------------------------------------------------
# CREATE PARSED DATAFRAME
# ------------------------------------------------------------

LLM_PARSED_DF = pd.DataFrame(
    PARSED_RESULTS
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_responses = len(
    LLM_PARSED_DF
)

successful_parses = int(
    (
        LLM_PARSED_DF[
            "parse_status"
        ]
        == "success"
    ).sum()
)

api_failures = int(
    (
        LLM_PARSED_DF[
            "api_status"
        ]
        != "success"
    ).sum()
)

invalid_json = int(
    (
        LLM_PARSED_DF[
            "parse_status"
        ]
        == "invalid_json"
    ).sum()
)

missing_fields = int(
    (
        LLM_PARSED_DF[
            "parse_status"
        ]
        == "missing_fields"
    ).sum()
)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

print()
print(
    f"Total responses       : {total_responses}"
)

print(
    f"Successfully parsed   : {successful_parses}"
)

print(
    f"API failures          : {api_failures}"
)

print(
    f"Invalid JSON          : {invalid_json}"
)

print(
    f"Missing fields        : {missing_fields}"
)

# ------------------------------------------------------------
# PARSE STATUS DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 90)
print("PARSE STATUS DISTRIBUTION")
print("=" * 90)

display(
    LLM_PARSED_DF[
        "parse_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "parse_status"
    )
    .reset_index(
        name="count"
    )
)

# ------------------------------------------------------------
# API ERROR DISTRIBUTION
# ------------------------------------------------------------

if api_failures > 0:

    print()
    print("=" * 90)
    print("API ERROR DISTRIBUTION")
    print("=" * 90)

    display(
        LLM_PARSED_DF[
            LLM_PARSED_DF[
                "api_status"
            ] != "success"
        ][
            [
                "dataset_id",
                "feature",
                "api_status",
                "api_error_type",
                "parse_status",
                "parse_error"
            ]
        ]
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

PARSED_OUTPUT_PATH = (
    "air_llm_parsed_recommendations.csv"
)

LLM_PARSED_DF.to_csv(
    PARSED_OUTPUT_PATH,
    index=False
)

print()
print(
    f"Saved parsed results to: "
    f"{PARSED_OUTPUT_PATH}"
)

print()
print("=" * 90)
print("NOTEBOOK 06.7 PARSING COMPLETE")
print("=" * 90)

NOTEBOOK 06.7 — PARSE STRUCTURED LLM RESPONSES

Total responses       : 9
Successfully parsed   : 0
API failures          : 9
Invalid JSON          : 0
Missing fields        : 0

PARSE STATUS DISTRIBUTION


,parse_status,count
0,not_parsed,9



API ERROR DISTRIBUTION


,dataset_id,feature,api_status,api_error_type,parse_status,parse_error
0,diabetes_130us,None,failed,quota_error,not_parsed,Error code: 429 - {'error': {'message': 'You e...
1,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
2,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
3,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
4,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
5,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
6,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
7,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...
8,diabetes_130us,None,not_attempted,quota_error,not_parsed,Skipped because account/API configuration prob...



Saved parsed results to: air_llm_parsed_recommendations.csv

NOTEBOOK 06.7 PARSING COMPLETE


In [23]:
# ============================================================
# 06.8 VALIDATE RECOMMENDATION FORMAT
# ============================================================

CANDIDATE_FIELDS = [
    "candidate_1",
    "candidate_2",
    "candidate_3",
    "candidate_4",
    "candidate_5"
]

VALIDATION_RECORDS = []

allowed_methods = set(
    METHOD_NAMES
)

for _, row in PARSED_LLM_DF.iterrows():

    parsed = row[
        "parsed_response"
    ]

    valid = True
    validation_errors = []

    # --------------------------------------------------------
    # PARSED OBJECT
    # --------------------------------------------------------

    if not isinstance(
        parsed,
        dict
    ):

        valid = False

        validation_errors.append(
            "response_not_valid_json"
        )

        VALIDATION_RECORDS.append({

            "dataset_id":
                row["dataset_id"],

            "feature":
                row["feature"],

            "valid":
                False,

            "errors":
                "; ".join(
                    validation_errors
                )
        })

        continue

    # --------------------------------------------------------
    # REQUIRED FIELDS
    # --------------------------------------------------------

    required_fields = [
        "feature",
        *CANDIDATE_FIELDS,
        "reason",
        "confidence"
    ]

    for field in required_fields:

        if field not in parsed:

            valid = False

            validation_errors.append(
                f"missing_{field}"
            )

    # --------------------------------------------------------
    # FEATURE
    # --------------------------------------------------------

    if (
        "feature" in parsed
        and str(
            parsed["feature"]
        ) != str(
            row["feature"]
        )
    ):

        valid = False

        validation_errors.append(
            "feature_mismatch"
        )

    # --------------------------------------------------------
    # CANDIDATES
    # --------------------------------------------------------

    candidates = []

    for field in CANDIDATE_FIELDS:

        if field not in parsed:
            continue

        method = str(
            parsed[field]
        ).strip()

        candidates.append(
            method
        )

        if method not in allowed_methods:

            valid = False

            validation_errors.append(
                f"invalid_method:{method}"
            )

    # --------------------------------------------------------
    # DUPLICATES
    # --------------------------------------------------------

    if len(candidates) != len(
        set(candidates)
    ):

        valid = False

        validation_errors.append(
            "duplicate_candidates"
        )

    # --------------------------------------------------------
    # CONFIDENCE
    # --------------------------------------------------------

    if "confidence" in parsed:

        try:

            confidence = float(
                parsed["confidence"]
            )

            if not (
                0.0 <= confidence <= 1.0
            ):

                valid = False

                validation_errors.append(
                    "confidence_out_of_range"
                )

        except Exception:

            valid = False

            validation_errors.append(
                "invalid_confidence"
            )

    # --------------------------------------------------------
    # REASON
    # --------------------------------------------------------

    if "reason" in parsed:

        reason = str(
            parsed["reason"]
        ).strip()

        if not reason:

            valid = False

            validation_errors.append(
                "empty_reason"
            )

    VALIDATION_RECORDS.append({

        "dataset_id":
            row["dataset_id"],

        "feature":
            row["feature"],

        "valid":
            valid,

        "errors":
            "; ".join(
                validation_errors
            )
    })

VALIDATION_DF = pd.DataFrame(
    VALIDATION_RECORDS
)

print("=" * 90)
print("RECOMMENDATION FORMAT VALIDATION")
print("=" * 90)

display(
    VALIDATION_DF
)

if not VALIDATION_DF[
    "valid"
].all():

    print(
        "\nWARNING: Some LLM responses "
        "failed validation."
    )

RECOMMENDATION FORMAT VALIDATION


,dataset_id,feature,valid,errors
0,diabetes_130us,race,False,response_not_valid_json
1,diabetes_130us,weight,False,response_not_valid_json
2,diabetes_130us,payer_code,False,response_not_valid_json
3,diabetes_130us,medical_specialty,False,response_not_valid_json
4,diabetes_130us,diag_1,False,response_not_valid_json
5,diabetes_130us,diag_2,False,response_not_valid_json
6,diabetes_130us,diag_3,False,response_not_valid_json
7,diabetes_130us,max_glu_serum,False,response_not_valid_json
8,diabetes_130us,A1Cresult,False,response_not_valid_json


In [24]:
# ============================================================
# 06.9 GENERATE RANKED TOP-K
# ============================================================

RANKED_RECOMMENDATIONS = []

for _, row in PARSED_LLM_DF.iterrows():

    parsed = row[
        "parsed_response"
    ]

    validation = VALIDATION_DF[
        (
            VALIDATION_DF[
                "dataset_id"
            ]
            == row["dataset_id"]
        )
        &
        (
            VALIDATION_DF[
                "feature"
            ]
            == row["feature"]
        )
    ]

    if validation.empty:
        continue

    if not bool(
        validation.iloc[0]["valid"]
    ):
        continue

    candidates = [
        str(
            parsed[field]
        ).strip()

        for field in CANDIDATE_FIELDS
    ]

    for rank, method in enumerate(
        candidates,
        start=1
    ):

        RANKED_RECOMMENDATIONS.append({

            "dataset_id":
                row["dataset_id"],

            "feature":
                row["feature"],

            "rank":
                int(rank),

            "method":
                method
        })

RANKED_RECOMMENDATIONS_DF = pd.DataFrame(
    RANKED_RECOMMENDATIONS
)

print("=" * 90)
print("RANKED TOP-K RECOMMENDATIONS")
print("=" * 90)

print(
    f"Valid features : "
    f"{RANKED_RECOMMENDATIONS_DF[['dataset_id', 'feature']].drop_duplicates().shape[0]}"
)

print(
    f"Recommendation rows : "
    f"{len(RANKED_RECOMMENDATIONS_DF)}"
)

display(
    RANKED_RECOMMENDATIONS_DF.head(25)
)

RANKED TOP-K RECOMMENDATIONS


KeyError: "None of [Index(['dataset_id', 'feature'], dtype='object')] are in the [columns]"

In [25]:
# ============================================================
# 06.10 — RECORD LLM RATIONALE
# ============================================================

import pandas as pd

print("=" * 90)
print("NOTEBOOK 06.10 — RECORD LLM RATIONALE")
print("=" * 90)

# ------------------------------------------------------------
# VALIDATE REQUIRED INPUT
# ------------------------------------------------------------

if "LLM_PARSED_DF" not in globals():

    raise RuntimeError(
        """
LLM_PARSED_DF is not available.

Run Notebook 06.7 — Parse Structured LLM Responses
before executing Notebook 06.10.
"""
    )

# ------------------------------------------------------------
# REQUIRED COLUMN
# ------------------------------------------------------------

if "reason" not in LLM_PARSED_DF.columns:

    raise RuntimeError(
        """
The 'reason' column is not available in LLM_PARSED_DF.

Check Notebook 06.7 and the structured output schema.
"""
    )

# ------------------------------------------------------------
# CREATE RATIONALE DATAFRAME
# ------------------------------------------------------------

LLM_RATIONALE_DF = LLM_PARSED_DF[
    [
        "experiment_index",
        "dataset_id",
        "feature",
        "api_status",
        "api_error_type",
        "parse_status",
        "reason"
    ]
].copy()

# ------------------------------------------------------------
# RATIONALE STATUS
# ------------------------------------------------------------

def determine_rationale_status(row):

    if row["api_status"] != "success":

        return "api_failure"

    if row["parse_status"] != "success":

        return "not_parsed"

    if pd.isna(row["reason"]):

        return "missing_rationale"

    if not str(
        row["reason"]
    ).strip():

        return "empty_rationale"

    return "available"


LLM_RATIONALE_DF[
    "rationale_status"
] = LLM_RATIONALE_DF.apply(
    determine_rationale_status,
    axis=1
)

# ------------------------------------------------------------
# RATIONALE LENGTH
# ------------------------------------------------------------

LLM_RATIONALE_DF[
    "rationale_length_chars"
] = (
    LLM_RATIONALE_DF[
        "reason"
    ]
    .fillna("")
    .astype(str)
    .str.len()
)

LLM_RATIONALE_DF[
    "rationale_word_count"
] = (
    LLM_RATIONALE_DF[
        "reason"
    ]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_features = len(
    LLM_RATIONALE_DF
)

available_rationales = int(
    (
        LLM_RATIONALE_DF[
            "rationale_status"
        ]
        == "available"
    ).sum()
)

missing_rationales = int(
    (
        LLM_RATIONALE_DF[
            "rationale_status"
        ].isin(
            [
                "missing_rationale",
                "empty_rationale"
            ]
        )
    ).sum()
)

api_failures = int(
    (
        LLM_RATIONALE_DF[
            "rationale_status"
        ]
        == "api_failure"
    ).sum()
)

not_parsed = int(
    (
        LLM_RATIONALE_DF[
            "rationale_status"
        ]
        == "not_parsed"
    ).sum()
)

# ------------------------------------------------------------
# PRINT RESULTS
# ------------------------------------------------------------

print()
print(
    f"Total features        : {total_features}"
)

print(
    f"Rationales available  : {available_rationales}"
)

print(
    f"Missing/empty         : {missing_rationales}"
)

print(
    f"API failures          : {api_failures}"
)

print(
    f"Not parsed            : {not_parsed}"
)

# ------------------------------------------------------------
# STATUS DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 90)
print("RATIONALE STATUS")
print("=" * 90)

display(
    LLM_RATIONALE_DF[
        "rationale_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "rationale_status"
    )
    .reset_index(
        name="count"
    )
)

# ------------------------------------------------------------
# DISPLAY AVAILABLE RATIONALES
# ------------------------------------------------------------

available_df = LLM_RATIONALE_DF[
    LLM_RATIONALE_DF[
        "rationale_status"
    ] == "available"
]

if not available_df.empty:

    print()
    print("=" * 90)
    print("LLM RATIONALES")
    print("=" * 90)

    display(
        available_df[
            [
                "dataset_id",
                "feature",
                "reason",
                "rationale_word_count"
            ]
        ]
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

RATIONALE_OUTPUT_PATH = (
    "air_llm_rationale.csv"
)

LLM_RATIONALE_DF.to_csv(
    RATIONALE_OUTPUT_PATH,
    index=False
)

print()
print(
    f"Rationale records saved to:"
)

print(
    RATIONALE_OUTPUT_PATH
)

print()
print("=" * 90)
print("NOTEBOOK 06.10 COMPLETE")
print("=" * 90)

NOTEBOOK 06.10 — RECORD LLM RATIONALE

Total features        : 9
Rationales available  : 0
Missing/empty         : 0
API failures          : 9
Not parsed            : 0

RATIONALE STATUS


,rationale_status,count
0,api_failure,9



Rationale records saved to:
air_llm_rationale.csv

NOTEBOOK 06.10 COMPLETE


In [26]:
# ============================================================
# 06.11 — RECORD LLM CONFIDENCE
# ============================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("NOTEBOOK 06.11 — RECORD LLM CONFIDENCE")
print("=" * 90)

# ------------------------------------------------------------
# VALIDATE INPUT
# ------------------------------------------------------------

if "LLM_PARSED_DF" not in globals():

    raise RuntimeError(
        """
LLM_PARSED_DF is not available.

Run Notebook 06.7 — Parse Structured LLM Responses
before executing Notebook 06.11.
"""
    )

# ------------------------------------------------------------
# REQUIRED COLUMN
# ------------------------------------------------------------

if "confidence" not in LLM_PARSED_DF.columns:

    raise RuntimeError(
        """
The 'confidence' column is not available.

Check Notebook 06.7 and the structured output schema.
"""
    )

# ------------------------------------------------------------
# CREATE CONFIDENCE DATAFRAME
# ------------------------------------------------------------

LLM_CONFIDENCE_DF = LLM_PARSED_DF[
    [
        "experiment_index",
        "dataset_id",
        "feature",
        "api_status",
        "api_error_type",
        "parse_status",
        "confidence"
    ]
].copy()

# ------------------------------------------------------------
# CONVERT TO NUMERIC
# ------------------------------------------------------------

LLM_CONFIDENCE_DF[
    "confidence"
] = pd.to_numeric(
    LLM_CONFIDENCE_DF[
        "confidence"
    ],
    errors="coerce"
)

# ------------------------------------------------------------
# CONFIDENCE STATUS
# ------------------------------------------------------------

def determine_confidence_status(row):

    if row["api_status"] != "success":

        return "api_failure"

    if row["parse_status"] != "success":

        return "not_parsed"

    value = row["confidence"]

    if pd.isna(value):

        return "missing"

    if not np.isfinite(value):

        return "non_finite"

    if value < 0:

        return "below_range"

    if value > 1:

        return "above_range"

    return "valid"


LLM_CONFIDENCE_DF[
    "confidence_status"
] = LLM_CONFIDENCE_DF.apply(
    determine_confidence_status,
    axis=1
)

# ------------------------------------------------------------
# NORMALIZED CONFIDENCE
# ------------------------------------------------------------

LLM_CONFIDENCE_DF[
    "confidence_normalized"
] = np.where(
    LLM_CONFIDENCE_DF[
        "confidence_status"
    ] == "valid",
    LLM_CONFIDENCE_DF[
        "confidence"
    ],
    np.nan
)

# ------------------------------------------------------------
# CONFIDENCE LEVEL
# ------------------------------------------------------------

def confidence_level(value):

    if pd.isna(value):

        return "unavailable"

    if value >= 0.80:

        return "high"

    if value >= 0.60:

        return "moderate"

    if value >= 0.40:

        return "low"

    return "very_low"


LLM_CONFIDENCE_DF[
    "confidence_level"
] = LLM_CONFIDENCE_DF[
    "confidence_normalized"
].apply(
    confidence_level
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_features = len(
    LLM_CONFIDENCE_DF
)

valid_confidence = int(
    (
        LLM_CONFIDENCE_DF[
            "confidence_status"
        ]
        == "valid"
    ).sum()
)

invalid_confidence = int(
    (
        ~LLM_CONFIDENCE_DF[
            "confidence_status"
        ].isin(
            [
                "valid",
                "api_failure",
                "not_parsed"
            ]
        )
    ).sum()
)

api_failures = int(
    (
        LLM_CONFIDENCE_DF[
            "confidence_status"
        ]
        == "api_failure"
    ).sum()
)

not_parsed = int(
    (
        LLM_CONFIDENCE_DF[
            "confidence_status"
        ]
        == "not_parsed"
    ).sum()
)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

print()
print(
    f"Total features       : {total_features}"
)

print(
    f"Valid confidence     : {valid_confidence}"
)

print(
    f"Invalid confidence   : {invalid_confidence}"
)

print(
    f"API failures         : {api_failures}"
)

print(
    f"Not parsed           : {not_parsed}"
)

# ------------------------------------------------------------
# CONFIDENCE STATUS
# ------------------------------------------------------------

print()
print("=" * 90)
print("CONFIDENCE STATUS")
print("=" * 90)

display(
    LLM_CONFIDENCE_DF[
        "confidence_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "confidence_status"
    )
    .reset_index(
        name="count"
    )
)

# ------------------------------------------------------------
# CONFIDENCE LEVEL DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 90)
print("CONFIDENCE LEVEL DISTRIBUTION")
print("=" * 90)

display(
    LLM_CONFIDENCE_DF[
        "confidence_level"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "confidence_level"
    )
    .reset_index(
        name="count"
    )
)

# ------------------------------------------------------------
# DISPLAY CONFIDENCE
# ------------------------------------------------------------

valid_confidence_df = LLM_CONFIDENCE_DF[
    LLM_CONFIDENCE_DF[
        "confidence_status"
    ] == "valid"
]

if not valid_confidence_df.empty:

    print()
    print("=" * 90)
    print("LLM CONFIDENCE")
    print("=" * 90)

    display(
        valid_confidence_df[
            [
                "dataset_id",
                "feature",
                "confidence",
                "confidence_level"
            ]
        ]
    )

# ------------------------------------------------------------
# SUMMARY STATISTICS
# ------------------------------------------------------------

if valid_confidence > 0:

    print()
    print("=" * 90)
    print("CONFIDENCE SUMMARY STATISTICS")
    print("=" * 90)

    print(
        f"Mean confidence   : "
        f"{valid_confidence_df['confidence'].mean():.4f}"
    )

    print(
        f"Median confidence : "
        f"{valid_confidence_df['confidence'].median():.4f}"
    )

    print(
        f"Minimum           : "
        f"{valid_confidence_df['confidence'].min():.4f}"
    )

    print(
        f"Maximum           : "
        f"{valid_confidence_df['confidence'].max():.4f}"
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

CONFIDENCE_OUTPUT_PATH = (
    "air_llm_confidence.csv"
)

LLM_CONFIDENCE_DF.to_csv(
    CONFIDENCE_OUTPUT_PATH,
    index=False
)

print()
print(
    f"Confidence records saved to:"
)

print(
    CONFIDENCE_OUTPUT_PATH
)

print()
print("=" * 90)
print("NOTEBOOK 06.11 COMPLETE")
print("=" * 90)

NOTEBOOK 06.11 — RECORD LLM CONFIDENCE

Total features       : 9
Valid confidence     : 0
Invalid confidence   : 0
API failures         : 9
Not parsed           : 0

CONFIDENCE STATUS


,confidence_status,count
0,api_failure,9



CONFIDENCE LEVEL DISTRIBUTION


,confidence_level,count
0,unavailable,9



Confidence records saved to:
air_llm_confidence.csv

NOTEBOOK 06.11 COMPLETE


In [27]:
# ============================================================
# 06.12 — SAVE AIR-LLM RECOMMENDATIONS
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 90)
print("NOTEBOOK 06.12 — SAVE AIR-LLM RECOMMENDATIONS")
print("=" * 90)

# ------------------------------------------------------------
# VALIDATE REQUIRED DATAFRAME
# ------------------------------------------------------------

if "LLM_PARSED_DF" not in globals():

    raise RuntimeError(
        """
LLM_PARSED_DF is not available.

Run Notebook 06.7 before executing Notebook 06.12.
"""
    )

# ------------------------------------------------------------
# REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [

    "experiment_index",
    "dataset_id",
    "feature",

    "api_status",
    "api_error_type",
    "parse_status",

    "candidate_1",
    "candidate_2",
    "candidate_3",
    "candidate_4",
    "candidate_5",

    "reason",
    "confidence"
]

missing_columns = [

    column
    for column in required_columns
    if column not in LLM_PARSED_DF.columns
]

if missing_columns:

    raise RuntimeError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )

# ------------------------------------------------------------
# CREATE FINAL RECOMMENDATION DATAFRAME
# ------------------------------------------------------------

AIR_LLM_RECOMMENDATIONS_DF = LLM_PARSED_DF[
    required_columns
].copy()

# ------------------------------------------------------------
# ADD RATIONALE STATUS
# ------------------------------------------------------------

def rationale_status(row):

    if row["api_status"] != "success":

        return "api_failure"

    if row["parse_status"] != "success":

        return "not_parsed"

    if pd.isna(row["reason"]):

        return "missing"

    if not str(
        row["reason"]
    ).strip():

        return "empty"

    return "available"


AIR_LLM_RECOMMENDATIONS_DF[
    "rationale_status"
] = AIR_LLM_RECOMMENDATIONS_DF.apply(
    rationale_status,
    axis=1
)

# ------------------------------------------------------------
# ADD CONFIDENCE STATUS
# ------------------------------------------------------------

AIR_LLM_RECOMMENDATIONS_DF[
    "confidence"
] = pd.to_numeric(
    AIR_LLM_RECOMMENDATIONS_DF[
        "confidence"
    ],
    errors="coerce"
)

def confidence_status(row):

    if row["api_status"] != "success":

        return "api_failure"

    if row["parse_status"] != "success":

        return "not_parsed"

    value = row["confidence"]

    if pd.isna(value):

        return "missing"

    if not np.isfinite(value):

        return "non_finite"

    if value < 0 or value > 1:

        return "out_of_range"

    return "valid"


AIR_LLM_RECOMMENDATIONS_DF[
    "confidence_status"
] = AIR_LLM_RECOMMENDATIONS_DF.apply(
    confidence_status,
    axis=1
)

# ------------------------------------------------------------
# ADD CONFIDENCE LEVEL
# ------------------------------------------------------------

def confidence_level(value):

    if pd.isna(value):

        return "unavailable"

    if value >= 0.80:

        return "high"

    if value >= 0.60:

        return "moderate"

    if value >= 0.40:

        return "low"

    return "very_low"


AIR_LLM_RECOMMENDATIONS_DF[
    "confidence_level"
] = AIR_LLM_RECOMMENDATIONS_DF[
    "confidence"
].apply(
    confidence_level
)

# ------------------------------------------------------------
# CHECK CANDIDATE STRATEGIES
# ------------------------------------------------------------

CANDIDATE_COLUMNS = [

    "candidate_1",
    "candidate_2",
    "candidate_3",
    "candidate_4",
    "candidate_5"
]

for column in CANDIDATE_COLUMNS:

    AIR_LLM_RECOMMENDATIONS_DF[
        column
    ] = (
        AIR_LLM_RECOMMENDATIONS_DF[
            column
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ------------------------------------------------------------
# COUNT AVAILABLE CANDIDATES
# ------------------------------------------------------------

AIR_LLM_RECOMMENDATIONS_DF[
    "candidate_count"
] = AIR_LLM_RECOMMENDATIONS_DF[
    CANDIDATE_COLUMNS
].apply(
    lambda row: sum(
        bool(value)
        for value in row
    ),
    axis=1
)

# ------------------------------------------------------------
# RECOMMENDATION STATUS
# ------------------------------------------------------------

def recommendation_status(row):

    if row["api_status"] != "success":

        return "api_failure"

    if row["parse_status"] != "success":

        return "parse_failure"

    if row["candidate_count"] < 5:

        return "incomplete_candidates"

    if row["rationale_status"] != "available":

        return "missing_rationale"

    if row["confidence_status"] != "valid":

        return "invalid_confidence"

    return "complete"


AIR_LLM_RECOMMENDATIONS_DF[
    "recommendation_status"
] = AIR_LLM_RECOMMENDATIONS_DF.apply(
    recommendation_status,
    axis=1
)

# ------------------------------------------------------------
# FINAL VALIDITY FLAG
# ------------------------------------------------------------

AIR_LLM_RECOMMENDATIONS_DF[
    "valid_recommendation"
] = (
    AIR_LLM_RECOMMENDATIONS_DF[
        "recommendation_status"
    ]
    == "complete"
)

# ------------------------------------------------------------
# COLUMN ORDER
# ------------------------------------------------------------

FINAL_COLUMN_ORDER = [

    "experiment_index",

    "dataset_id",

    "feature",

    "candidate_1",
    "candidate_2",
    "candidate_3",
    "candidate_4",
    "candidate_5",

    "candidate_count",

    "reason",

    "rationale_status",

    "confidence",

    "confidence_level",

    "confidence_status",

    "api_status",

    "api_error_type",

    "parse_status",

    "recommendation_status",

    "valid_recommendation"
]

AIR_LLM_RECOMMENDATIONS_DF = (
    AIR_LLM_RECOMMENDATIONS_DF[
        FINAL_COLUMN_ORDER
    ]
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_records = len(
    AIR_LLM_RECOMMENDATIONS_DF
)

complete_records = int(
    AIR_LLM_RECOMMENDATIONS_DF[
        "valid_recommendation"
    ].sum()
)

incomplete_records = (
    total_records
    - complete_records
)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

print()
print("=" * 90)
print("AIR-LLM RECOMMENDATION SUMMARY")
print("=" * 90)

print(
    f"Total feature records : {total_records}"
)

print(
    f"Complete records      : {complete_records}"
)

print(
    f"Incomplete records    : {incomplete_records}"
)

# ------------------------------------------------------------
# STATUS DISTRIBUTION
# ------------------------------------------------------------

print()
print("=" * 90)
print("RECOMMENDATION STATUS")
print("=" * 90)

display(
    AIR_LLM_RECOMMENDATIONS_DF[
        "recommendation_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "recommendation_status"
    )
    .reset_index(
        name="count"
    )
)

# ------------------------------------------------------------
# DISPLAY FINAL RECOMMENDATIONS
# ------------------------------------------------------------

if complete_records > 0:

    print()
    print("=" * 90)
    print("VALID AIR-LLM RECOMMENDATIONS")
    print("=" * 90)

    display(
        AIR_LLM_RECOMMENDATIONS_DF[
            AIR_LLM_RECOMMENDATIONS_DF[
                "valid_recommendation"
            ]
        ][
            [
                "dataset_id",
                "feature",
                "candidate_1",
                "candidate_2",
                "candidate_3",
                "candidate_4",
                "candidate_5",
                "reason",
                "confidence",
                "confidence_level"
            ]
        ]
    )

else:

    print()
    print(
        "WARNING: No complete AIR-LLM "
        "recommendations are currently available."
    )

    print(
        "This is expected while the OpenAI API "
        "quota is exhausted."
    )

# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

RECOMMENDATION_CSV_PATH = (
    "air_llm_recommendations.csv"
)

AIR_LLM_RECOMMENDATIONS_DF.to_csv(
    RECOMMENDATION_CSV_PATH,
    index=False
)

# ------------------------------------------------------------
# SAVE EXCEL
# ------------------------------------------------------------

RECOMMENDATION_XLSX_PATH = (
    "air_llm_recommendations.xlsx"
)

with pd.ExcelWriter(
    RECOMMENDATION_XLSX_PATH,
    engine="openpyxl"
) as writer:

    AIR_LLM_RECOMMENDATIONS_DF.to_excel(
        writer,
        sheet_name="Recommendations",
        index=False
    )

    # --------------------------------------------------------
    # SUMMARY SHEET
    # --------------------------------------------------------

    summary_df = pd.DataFrame({

        "metric": [

            "Total feature records",
            "Complete recommendations",
            "Incomplete recommendations"

        ],

        "value": [

            total_records,
            complete_records,
            incomplete_records

        ]
    })

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    # --------------------------------------------------------
    # STATUS SHEET
    # --------------------------------------------------------

    status_df = (
        AIR_LLM_RECOMMENDATIONS_DF[
            "recommendation_status"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "recommendation_status"
        )
        .reset_index(
            name="count"
        )
    )

    status_df.to_excel(
        writer,
        sheet_name="Status",
        index=False
    )

# ------------------------------------------------------------
# SAVE VALID-ONLY DATASET
# ------------------------------------------------------------

VALID_RECOMMENDATIONS_DF = (
    AIR_LLM_RECOMMENDATIONS_DF[
        AIR_LLM_RECOMMENDATIONS_DF[
            "valid_recommendation"
        ]
    ].copy()
)

VALID_RECOMMENDATIONS_PATH = (
    "air_llm_valid_recommendations.csv"
)

VALID_RECOMMENDATIONS_DF.to_csv(
    VALID_RECOMMENDATIONS_PATH,
    index=False
)

# ------------------------------------------------------------
# FINAL OUTPUT INFORMATION
# ------------------------------------------------------------

print()
print("=" * 90)
print("FILES SAVED")
print("=" * 90)

print(
    f"CSV              : "
    f"{RECOMMENDATION_CSV_PATH}"
)

print(
    f"Excel            : "
    f"{RECOMMENDATION_XLSX_PATH}"
)

print(
    f"Valid-only CSV   : "
    f"{VALID_RECOMMENDATIONS_PATH}"
)

print()
print("=" * 90)
print("NOTEBOOK 06.12 COMPLETE")
print("=" * 90)

NOTEBOOK 06.12 — SAVE AIR-LLM RECOMMENDATIONS

AIR-LLM RECOMMENDATION SUMMARY
Total feature records : 9
Complete records      : 0
Incomplete records    : 9

RECOMMENDATION STATUS


,recommendation_status,count
0,api_failure,9



This is expected while the OpenAI API quota is exhausted.

FILES SAVED
CSV              : air_llm_recommendations.csv
Excel            : air_llm_recommendations.xlsx
Valid-only CSV   : air_llm_valid_recommendations.csv

NOTEBOOK 06.12 COMPLETE
